<div align="center">
<img src="docs/Enamel_Brooch.png" width="120" style="margin-bottom: 10px;">

# GeoSAR-Forge — PS/DS-InSAR 地表形变自动化监测系统

**Sentinel-1 多时相 InSAR 主链 + support_graph_v1 区域识别 + MintPy 后未来预测 v2**

`ISCE2` 几何配准 → `Dolphin` PS/DS 相位连接 → `主链 QC / DePSI-like QC` → `MintPy` 两阶段时序反演  
`MintPy` 最终接受结果 → `support_graph_v1` 异常形变区识别 → `generic / hazard` 双模式未来预测

---</div>

## 当前正式主线

- 主链：`AOI / 场景筛选 → 大气方案 → DEM → SLC 下载 → ISCE2 → Dolphin → Mainchain QC / DePSI-like QC → MintPy(pass1→feedback→pass2)`
- 区域识别：默认 `support_graph_v1`，在 `strict ∪ relaxed` 支持点上做图分割、候选区评分和 outward growth
- 预测链：默认 `forecast_mode="generic"`，并行训练 `TemporalFusionForecaster` 与 `DecompTCNGRUQuantileForecaster` 后按验证集 MAE 选优；`hazard` 用 `GraphTCNAttnQuantileForecaster`
- 不确定性：正式输出采用 `cqr_conformal_v1` 校准区间，`c_pred` 是 calibrated interval 驱动的 risk-aware surrogate

### 技术流程

<div align="center">
<img src="docs/pipeline_flowchart.png" width="980" style="margin: 10px 0;">
</div>

更多版本：[`SVG`](/data/InSAR/docs/pipeline_flowchart.svg) / [`高清 PNG`](/data/InSAR/docs/pipeline_flowchart_hires.png) / [`Mermaid 源文件`](/data/InSAR/docs/pipeline_flowchart.mmd)

### 文档入口

- [当前正式流程说明](/data/InSAR/docs/PIPELINE.md)
- [DePSI-like QC 创新说明](/data/InSAR/docs/DePSI_like_QC_创新说明.md)
- [MintPy 后未来预测说明](/data/InSAR/docs/MintPy后高可信点预测说明.md)
- [用户手册](/data/InSAR/docs/用户手册.md)


## Cell 1: 项目初始化与全局配置

### 项目隔离机制

每个处理任务创建独立项目目录，所有数据文件隔离存放，防止不同区域/时段的数据"串味":

```
/data/InSAR/projects/
├── 兰州_2022_2024/          ← 项目1
│   ├── project.json         ← 项目配置+进度 (断点恢复)
│   ├── SLC_zip/
│   ├── isce2/
│   ├── dolphin_work/
│   ├── mintpy/
│   ├── export/
│   └── pipeline.log
├── 昆明呈贡_2021/           ← 项目2
│   └── ...
```

- **新建项目**: 输入名称，自动创建目录结构
- **恢复项目**: 选择已有项目编号，从上次进度继续
- **自动清理**: 可选开启/关闭中间文件删除 (节省磁盘 vs 保留调试数据)

> **AOI 支持三种输入**:
> 1. 行政区名称列表: `["城关区", "七里河区"]`
> 2. 单个中文地名: `"兰州市"`
> 3. NESW 边界: `{"N": 36.2, "S": 35.8, "W": 103.5, "E": 104.1}`

In [ ]:
import sys
sys.path.insert(0, '/data/InSAR')

from insar_utils.config import *

setup_proxy()    # 配置 http/https/socks5 代理 (127.0.0.1:7890)

# ============================================================
#  项目初始化 — 每个项目独立目录, 防止数据"串味"
#  已有项目: 选择编号恢复 | 新项目: 输入名称创建
#  同时选择是否启用中间文件自动清理
# ============================================================
project_name = init_project()

# 重新导入更新后的路径变量 (init_project 会修改所有路径)
from insar_utils.config import *

setup_logging()  # 日志: DEBUG/INFO → pipeline.log, WARNING/ERROR → 页面
disk_check()     # 检查 /data 剩余空间

# ============================================================
#  用户配置区 —— 运行前请根据实际需求修改以下参数
# ============================================================
CONFIG = {
    # 研究区域 (AOI) — 三种输入方式:
    # 方式一 (推荐): 行政区名称列表，支持省/市/县(区)级
    "aoi": ["呈贡区"],
    # 方式二: 单个中文地名
    # "aoi": "兰州市",
    # 方式三: 直接指定边界
    # "aoi": {"N": 36.2, "S": 35.8, "W": 103.5, "E": 104.1},

    "aoi_buffer_deg": 0.05,

    # 时间范围
    "date_range": ("2021-01-01", "2021-12-31"),

    # 场景选择
    "target_scenes": 20,
    "flight_direction": "DESCENDING",
}

print(f"\n项目 [{project_name}] 初始化完成。修改上方 CONFIG 后按顺序执行。")
print(f"项目目录: {WORK_DIR}")

## Cell 2: 行政区划选片与场景筛选

这一单元只负责 **AOI 解析、ASF 搜索与时间均匀采样**，不再在这里直接下载 SLC。这样我们可以在确定待处理场景后，先完成大气校正方案选择，再进入 SLC 下载与主链反演。

### 时间均匀采样算法

当可用场景数 $N_{total}$ 超过目标数 $N_{target}$ 时，系统在时间轴上做均匀抽样，并兼顾垂直基线分布：

$$
\Delta t_{ideal} = \frac{t_{end}-t_{start}}{N_{target}-1}
$$

对每个理想时隙 $t_i$，选择综合得分最优的场景：

$$
score(j)=|t_j-t_i| + w\cdot|B_{\perp,j}-\tilde B_\perp|
$$

其中 $\tilde B_\perp$ 为中位数垂直基线，$w$ 为时间与基线偏差之间的平衡权重。这样做的目的，是在保证时序均匀性的同时，避免后续网络里出现过于极端的空间基线。

### 当前单元的输出

- `aoi_wkt / bounds`
- `scenes`
- `selected`

后面的 `Cell 3` 会基于 `selected` 先确定大气校正方案，再进入 SLC 下载。


In [ ]:
from insar_utils.downloader import (
    resolve_aoi, search_scenes, uniform_temporal_sample,
)
from insar_utils.config import *
from insar_utils.config import save_project_progress

# 第一步: 解析行政区划 → 人工复审 → 连续性检查
aoi_wkt, bounds = resolve_aoi(
    CONFIG["aoi"],
    buffer_deg=CONFIG.get("aoi_buffer_deg", 0.05)
)

# 第二步: ASF 搜索 Sentinel-1 SLC
scenes = search_scenes(
    aoi_wkt,
    CONFIG["date_range"],
    CONFIG.get("flight_direction", "DESCENDING")
)

# 第三步: 时间均匀采样 (贪心算法 + 基线优化)
selected = uniform_temporal_sample(scenes, CONFIG["target_scenes"])

# 当前阶段只完成场景筛选，不下载 SLC
print(f"已筛选场景数: {len(selected)}")
save_project_progress("scene_selection_done", aoi=str(CONFIG["aoi"]))


## Cell 3: 大气方案选择与 DEM 准备

这一单元现在位于 **SLC 下载之前**。原因是大气校正方案已经成为主链配置的一部分，应当在项目开始阶段就写入 `project.json`，后续由主链 QC 与 MintPy 统一消费，而不是等到 SLC 下载完成后再补充。

### 大气校正方案

当前支持：

- `GACOS`
- `ERA5`
- `MERRA-2`
- `height_correlation`
- `spatial_filter`
- `no`

其中 `GACOS / ERA5 / MERRA-2` 的数据准备都放在 MintPy 前的主链 QC 里统一处理；这里的作用只是 **记录项目方案**。`GACOS` 尤其如此：当前阶段不再给出最终下载范围，而是在 MintPy 前按实际 Dolphin 网格和子窗口生成精确下载指引，尽量保证只下载一次。

### DEM

DEM 仍在这一阶段准备，因为后续 SLC 下载完成后，`ISCE2` 需要立刻使用 DEM 做几何配准与地形相位准备。


In [ ]:
from insar_utils.dem import download_srtm_dem
from insar_utils.atmosphere import choose_atmo_correction
from insar_utils.config import *

# 第一步: 先确定大气校正方案（早于 SLC 下载）
atmo_config = choose_atmo_correction(selected, bounds)

# 第二步: 下载 SRTM 30m DEM
download_srtm_dem(bounds, DEM_DIR, OPENTOPO_KEY)

print(f"当前大气方案: {atmo_config['method']}")


## Cell 4: SLC 下载、精密轨道与 ISCE2 几何配准

这一单元从这里开始才真正进入 **SLC 下载**，因此当前主线顺序已经变成：

1. AOI / 场景筛选
2. 大气方案选择
3. SLC 下载
4. ISCE2 几何配准

这更符合项目级流程控制的思路：大气方案先写入配置，再让后续主链统一消费。

同时这一单元也继续负责：

- 精密轨道下载
- SLC 解压
- `stackSentinel.py` 生成 ISCE2 运行脚本
- 合并与裁剪后的 SLC 后处理

当前 `postprocess_slc()` 这里指的是 **ISCE2 合并后的栅格修复与裁剪**，不是旧的深度学习后处理链。


In [ ]:
from insar_utils.downloader import create_asf_session, download_all
from insar_utils.isce_runner import (
    prepare_isce_dirs, download_orbits, extract_slc_zips,
    generate_stack, run_isce_steps, postprocess_slc,
)
from insar_utils.config import *

# 第一步: 下载筛选后的 SLC
session = create_asf_session()
download_all(selected, SLC_ZIP_DIR, session, parallel=4)

# 第二步: 创建 ISCE2 目录结构
prepare_isce_dirs(ISCE_WORK_DIR)

# 第三步: 下载精密轨道文件 (EOF)
download_orbits(selected, ORBIT_DIR)

# 第四步: 并行解压 SLC ZIP → .SAFE 目录
extract_slc_zips(SLC_ZIP_DIR, ISCE_SLC_DIR)
cleanup_after_step("after_extract")

# 第五步: 生成配准脚本 — 传入 AOI bbox 自动裁剪
aoi_bbox = [bounds["S"], bounds["N"], bounds["W"], bounds["E"]]
generate_stack(ISCE_SLC_DIR, DEM_PATH, ORBIT_DIR, ISCE_WORK_DIR, bbox=aoi_bbox)

# 第六步: 执行 ISCE2 几何配准
run_isce_steps(RUN_DIR, LOG_FILE)

# 第七步: ISCE2 合并后的栅格修复、GeoTIFF 转换与裁剪
postprocess_slc(ISCE_WORK_DIR)
cleanup_after_step("after_isce2")
cleanup_after_step("after_postprocess")

save_project_progress("isce2_done")


## Cell 5: Dolphin PS/DS 相位连接与并行解缠

### PS 与 DS 散射体

- **PS (Persistent Scatterer)**: 振幅稳定的强散射体 (建筑角反射器、岩石露头)。判据: 振幅离差指数 $D_A = \sigma_A / \bar{A} < 0.4$
- **DS (Distributed Scatterer)**: 空间上均匀分布的弱散射体 (裸土、草地)。其相干性通过空间邻域统计估计提升

### Phase Linking 算法 (EVD/EMI)

对 $N$ 景 SLC 构成的时间序列，每个像素的协方差矩阵为:

$$\mathbf{C} = \frac{1}{L} \sum_{l=1}^{L} \mathbf{s}_l \mathbf{s}_l^H \in \mathbb{C}^{N \times N}$$

其中 $L$ 为空间邻域像素数（由 `half_window` 控制，本流水线设为 11×5 = 110 个像素），$\mathbf{s}_l = [s_1, s_2, ..., s_N]^T$ 为第 $l$ 个邻域像素的 SLC 向量。

**EMI (Eigenvalue Maximization of Interferometric phase)** 估计最优相位:

$$\hat{\boldsymbol{\phi}} = \arg\max_{\boldsymbol{\phi}} \boldsymbol{\phi}^H \cdot |\mathbf{C}| \circ e^{j \angle \mathbf{C}} \cdot \boldsymbol{\phi}$$

等价于对相干性矩阵 $\boldsymbol{\Gamma} = \mathbf{D}^{-1/2} \mathbf{C} \mathbf{D}^{-1/2}$ (其中 $\mathbf{D} = \text{diag}(\mathbf{C})$) 做特征值分解 (EVD)，取最大特征值对应的特征向量。

### 多视 (Strides)

`strides = {x:6, y:3}` 意味着在距离向和方位向分别以 6:3 比例下采样。多视等效视数:

$$L_{eq} = \text{strides}_x \times \text{strides}_y \times L_{window} = 6 \times 3 \times 110 = 1980$$

### SNAPHU 相位解缠

Phase Linking 输出的相位仍是缠绕的 ($\phi \in [-\pi, \pi]$)。SNAPHU 通过最小化全局代价函数进行解缠:

$$\hat{\boldsymbol{\psi}} = \arg\min_{\boldsymbol{\psi}} \sum_{(i,j) \in \mathcal{E}} c_{ij} \cdot |(\psi_i - \psi_j) - (\phi_i - \phi_j)|$$

其中 $\boldsymbol{\psi}$ 为解缠相位，$\mathcal{E}$ 为像素邻接图的边集，$c_{ij}$ 为代价权重 (与相干性相关)。

这是一个网络流优化问题，SNAPHU 使用最小费用流 (MCF) 算法求解。本流水线配置 `n_parallel_jobs=5` 路并行解缠。

In [ ]:
from insar_utils.dolphin_runner import build_dolphin_config, run_dolphin
from insar_utils.config import *  # 确保路径变量最新

# 第一步: 构建 Dolphin 配置 (SLC GeoTIFF 输入)
slc_tif_pattern = str(ISCE_WORK_DIR / "merged" / "SLC" / "*" / "*.slc.tif")
cfg_path = build_dolphin_config(
    slc_pattern=slc_tif_pattern,
    work_dir=DOLPHIN_DIR,
    n_workers=N_WORKERS,
)

# 第二步: 执行 Phase Linking + 并行解缠 (Python API, 自动 PS/DS 分离)
run_dolphin(cfg_path, LOG_FILE, n_workers=N_WORKERS)
# 注意: pre-MintPy QC 仍需使用 phase_linking 中的 temporal coherence / PS 辅助栅格，
# 不要在这里执行 after_dolphin 清理。桥接完成后由 after_hdf5_bridge 统一清理。

save_project_progress("dolphin_done")

## Cell 5A: 主链 QC、DePSI-like 质量评分与 MintPy 前预检

这一单元位于 Dolphin 解缠之后、MintPy 之前，负责把质量控制前移到时序反演之前。当前不仅会生成 `pair_qc.csv`、`pair_qc_summary.json` 和 `pair_qc_matrix.png`，还会额外构建一套 DePSI-like QC 产物：`ps_score.tif`、`mask_ps_strict.tif`、`mask_ps_relaxed.tif`、`ref_primary_network.csv/.tif`、`ref_candidates.csv` 和 `ps_score_summary.json`。其中 `ps_score` 借鉴的是更保守的 PSI 思想，不只看单一相干阈值，而是综合 temporal coherence、有效 pair 比例、主连通分量占比、时间模型残差和 jump 风险来筛出高可信 PS 骨架。

当前 `ps_score` 的核心定义是：

$$
ps\_score = 0.30\,tcoh + 0.20\,valid\_pair\_ratio + 0.20\,mainCC\_ratio + 0.15(1-model\_rms\_norm) + 0.15(1-jump\_risk)
$$

这里的 `model_rms_norm` 来自 Dolphin 时序的轻量时间模型拟合，`jump_risk` 则来自 pair 级 residual 的自适应 jump 判据，因此这一步既看空间质量，也看时间一致性。`strict / relaxed` 阈值保留默认值，但系统会在 `ps_score_summary.json` 中同时写出覆盖率和告警，避免简单把阈值当成全项目固定常数。

当前参考点也不再按“单像元最高相干”去挑，而是先在 strict PS 中建立一个空间上分散、质量更高的参考主网络，再从这个网络里排出 top N 候选，并为每个候选统计其周围 `3×3/5×5` 小参考区的稳定性。后面的 MintPy 第一轮会优先从这些候选中选点；如果 top1 在 pass1 后表现异常，就会自动回退到 top2/top3，而不是把整个结果押注在一个孤立像元上。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from insar_utils.viz import display_notebook_figure_gallery
from insar_utils.mainchain_qc import run_pre_mintpy_qc, run_dolphin_ablation
from insar_utils.config import *  # 确保路径变量最新

MAINCHAIN_QC_CONFIG = {
    "enable_pair_qc": True,
    "pair_qc_mode": "drop+downweight",
    "gacos_coverage_interactive": True,
    "gacos_manual_wait": True,
    "gacos_min_coverage_warn": 0.95,
    "gacos_min_coverage_ok": 0.99,
    "gacos_low_coverage_fallback": "height_correlation",
    "padding_candidates": [20, 10, 0],
    "enable_dolphin_ablation": False,
}

DEPSI_LIKE_QC_CONFIG = {
    "strict_ps_threshold": 0.75,
    "relaxed_ps_threshold": 0.60,
    "jump_threshold_base_mm": 28.0,
    "jump_threshold_quantile": 0.95,
    "jump_threshold_scale": 1.0,
    "abnormal_date_abs_frac_threshold": 0.20,
    "abnormal_date_abs_residual_mm": 20.0,
    "abnormal_date_quantile": 0.90,
    "ref_candidate_top_n": 20,
    "ref_candidate_patch_radius": 2,
    "ref_network_min_distance_m": 400.0,
    "ref_min_tcoh": 0.70,
    "ref_min_valid_pair_ratio": 0.80,
    "ref_min_maincc_ratio": 0.80,
}

mainchain_qc_summary = run_pre_mintpy_qc(
    dolphin_dir=DOLPHIN_DIR,
    geom_source_dir=ISCE_WORK_DIR / "merged" / "geom_reference",
    mintpy_dir=MINTPY_DIR,
    gacos_dir=GACOS_DIR,
    atmo_config=atmo_config,
    report_dir=WORK_DIR / "mainchain_qc",
    interactive=MAINCHAIN_QC_CONFIG["gacos_coverage_interactive"],
    manual_wait=MAINCHAIN_QC_CONFIG["gacos_manual_wait"],
    gacos_min_coverage_warn=MAINCHAIN_QC_CONFIG["gacos_min_coverage_warn"],
    gacos_min_coverage_ok=MAINCHAIN_QC_CONFIG["gacos_min_coverage_ok"],
    padding_candidates=MAINCHAIN_QC_CONFIG["padding_candidates"],
    low_coverage_fallback=MAINCHAIN_QC_CONFIG["gacos_low_coverage_fallback"],
    depsi_like_config=DEPSI_LIKE_QC_CONFIG,
)


def _format_bounds(bounds):
    if not bounds:
        return "N/A"
    return (
        f"N={float(bounds['N']):.4f}, S={float(bounds['S']):.4f}, "
        f"W={float(bounds['W']):.4f}, E={float(bounds['E']):.4f}"
    )


display(mainchain_qc_summary)

scene_dates = list(mainchain_qc_summary.get("scene_dates") or [])
if scene_dates:
    print("\n===== GACOS 目标日期（竖排，可直接复制） =====\n")
    print("\n".join(scene_dates))

utc_hms = mainchain_qc_summary.get("acquisition_utc_hms")
required_bounds = mainchain_qc_summary.get("gacos_redownload_required_bounds")
recommended_bounds = mainchain_qc_summary.get("gacos_redownload_recommended_bounds")
guide_path = Path(mainchain_qc_summary.get("gacos_download_guide", "")) if mainchain_qc_summary.get("gacos_download_guide") else None
if guide_path and guide_path.exists():
    print("\n===== GACOS 下载提示 =====\n")
    if utc_hms:
        print(f"UTC 时间: {utc_hms}")
    if required_bounds:
        print(f"最低必要范围: {_format_bounds(required_bounds)}")
    if recommended_bounds:
        print(f"稳妥推荐范围: {_format_bounds(recommended_bounds)}")
    print(f"下载指南文件: {guide_path}")

ps_score_summary_json = Path(mainchain_qc_summary.get("ps_score_summary_json", "")) if mainchain_qc_summary.get("ps_score_summary_json") else None
if ps_score_summary_json and ps_score_summary_json.exists():
    ps_score_summary = json.loads(ps_score_summary_json.read_text(encoding="utf-8"))
    print("\n===== DePSI-like QC 摘要 =====\n")
    display({
        "strict_coverage_ratio": round(float(ps_score_summary.get("strict_coverage_ratio", 0.0)), 6),
        "relaxed_coverage_ratio": round(float(ps_score_summary.get("relaxed_coverage_ratio", 0.0)), 6),
        "strict_candidate_count": int(ps_score_summary.get("strict_candidate_count", 0)),
        "primary_network_count": int(ps_score_summary.get("primary_network_count", 0)),
        "coverage_warning_flag": bool(ps_score_summary.get("coverage_warning_flag", False)),
        "coverage_warning_reason": ps_score_summary.get("coverage_warning_reason", []),
        "selection_fallback_reason": ps_score_summary.get("selection_fallback_reason", ""),
        "jump_threshold_applied_mm": round(float(ps_score_summary.get("jump_threshold_applied_mm", 0.0)), 3),
        "strict_ps_residual_p90_mm": round(float(ps_score_summary.get("strict_ps_residual_p90_mm", 0.0)), 3),
        "strict_ps_residual_p95_mm": round(float(ps_score_summary.get("strict_ps_residual_p95_mm", 0.0)), 3),
        "ref_network_min_distance_m": round(float(ps_score_summary.get("ref_network_min_distance_m", 0.0)), 1),
        "ref_candidate_patch_radius": int(ps_score_summary.get("ref_candidate_patch_radius", 0)),
    })

ref_candidates_csv = Path(mainchain_qc_summary.get("ref_candidates_csv", "")) if mainchain_qc_summary.get("ref_candidates_csv") else None
if ref_candidates_csv and ref_candidates_csv.exists():
    ref_df = pd.read_csv(ref_candidates_csv)
    print("\n===== 参考点候选（前 10 条） =====\n")
    display(ref_df.head(10))

ref_network_csv = Path(mainchain_qc_summary.get("ref_primary_network_csv", "")) if mainchain_qc_summary.get("ref_primary_network_csv") else None
if ref_network_csv and ref_network_csv.exists():
    ref_network_df = pd.read_csv(ref_network_csv)
    print("\n===== 参考主网络（前 10 条） =====\n")
    display(ref_network_df.head(10))

scene_qc_csv = Path(mainchain_qc_summary.get("scene_qc_csv", "")) if mainchain_qc_summary.get("scene_qc_csv") else None
if scene_qc_csv and scene_qc_csv.exists():
    scene_qc_df = pd.read_csv(scene_qc_csv)
    for col in ["mean_pair_risk", "max_pair_risk", "mean_pair_weight"]:
        if col in scene_qc_df.columns:
            scene_qc_df[col] = scene_qc_df[col].map(lambda v: round(float(v), 4))
    print("\n===== 每景 Pair 风险分数与权重系数 =====\n")
    display(scene_qc_df)

matrix_path = Path(mainchain_qc_summary.get("pair_qc_matrix_png", "")) if mainchain_qc_summary.get("pair_qc_matrix_png") else None
if matrix_path and matrix_path.exists():
    display_notebook_figure_gallery({"Pair QC Matrix": matrix_path}, title="Pair QC Matrix", columns=1, max_height_px=480)

if MAINCHAIN_QC_CONFIG["enable_dolphin_ablation"]:
    dolphin_ablation_summary = run_dolphin_ablation(
        work_root=WORK_DIR / "dolphin_ablation",
        slc_pattern=str(SLC_DIR / "*.slc.tif"),
        geom_source_dir=ISCE_WORK_DIR / "merged" / "geom_reference",
        atmo_config=atmo_config,
        enabled=True,
    )
    display(dolphin_ablation_summary)

save_project_progress("mainchain_qc_done")


## Cell 6: MintPy 两阶段时序反演、反馈回灌与误差校正

这一单元只负责 **MintPy 两阶段反演本身**：先构建 `ifgramStack_original.h5 / geometryRadar.h5`，再执行 `pass1 → feedback → pass2` 的反馈回灌流程，并决定最终接受 `pass1` 还是 `pass2`。

这里不再混入导出与绘图逻辑，避免主反演和成果展示混在一起。


In [ ]:
from IPython.display import display
from insar_utils.mintpy_runner import generate_mintpy_template, build_mintpy_hdf5
from insar_utils.mainchain_qc import run_mintpy_feedback_roundtrip
from insar_utils.config import *  # 确保路径变量最新

# MintPy 两阶段反演
# 1. 构建 Dolphin→MintPy 桥接输入
build_mintpy_hdf5(
    DOLPHIN_DIR,
    ISCE_WORK_DIR / "merged" / "geom_reference",
    MINTPY_DIR,
    ifgram_filename="ifgramStack_original.h5",
)
cleanup_after_step("after_hdf5_bridge")

# 2. 生成 MintPy 模板
tpl = generate_mintpy_template(
    dolphin_dir=DOLPHIN_DIR,
    dem_file=DEM_FILE,
    era5_dir=ERA5_DIR,
    gacos_dir=GACOS_DIR,
    output_path=TEMPLATE_PATH,
    atmo_config=atmo_config,
)

# 3. 执行 pass1 → feedback → pass2
mintpy_feedback_summary = run_mintpy_feedback_roundtrip(
    template_path=tpl,
    mintpy_dir=MINTPY_DIR,
    geom_source_dir=ISCE_WORK_DIR / "merged" / "geom_reference",
    log_file=LOG_FILE,
    report_dir=WORK_DIR / "mainchain_qc",
    depsi_like_config=DEPSI_LIKE_QC_CONFIG,
    source_ifgram_name="ifgramStack_original.h5",
    qc_ifgram_name="ifgramStack_qc.h5",
)
display(mintpy_feedback_summary)

save_project_progress("mintpy_done")


## Cell 7: 标准化可视化与成果导出

这一单元负责 **标准化科研绘图、成果导出和结果预览**。图件参数参考 Nature/RSE/JGR 等科研期刊常见版式：

### 出图规范
- **字体**: Arial/Helvetica 无衬线, 8pt 正文 / 7pt 刻度
- **尺寸**: 单栏 89 mm / 双栏 183 mm (Nature 系标准)
- **色标**: RdBu_r (速率, 0 居中), YlOrRd (不确定性), inferno (相干性)
- **格式**: 同时输出 300 dpi PNG + PDF 矢量版, 可直接投稿

### 输出图件
1. **基线网络图** — 时空紧密度着色, 内嵌相干性直方图
2. **速率图** — 双面板: (a) LOS velocity + DEM hillshade, (b) Velocity std; 含比例尺/北向箭头/区位 inset/速率直方图
3. **速率诊断** — 四联图: vel-height / vel-coherence / 速率分布 / 相干性空间分布
4. **位移面板** — 子图标签 (a)-(i), 精选 + 全时相两版
5. **时序图** — 多代表点自动选取, 线性拟合趋势线 + 速率标注
6. **导出** — GeoTIFF / Shapefile / KMZ / CSV
7. **高可信成果** — DePSI-like 高可信点分层导出

In [ ]:
%matplotlib inline
import matplotlib
from pathlib import Path
from IPython.display import display

# rcParams 已由 viz.py 模块自动设置为标准化科研绘图参数（参考 Nature/RSE/JGR 常见版式）
# 此处仅覆盖 Jupyter 特有项
matplotlib.rcParams.update({
    "figure.dpi": 150,
})

from insar_utils.viz import (
    plot_velocity_map, plot_point_timeseries, plot_displacement_panels,
    plot_baseline_network, plot_velocity_diagnostics,
    export_geotiff, export_shapefile, export_kmz, export_csv,
    display_notebook_figure_gallery, organize_export_dir,
)
from insar_utils.mainchain_qc import export_depsi_like_outputs
from insar_utils.config import *  # 确保路径变量最新

# ============================================================================
# 标准化可视化与成果导出
# ============================================================================

# 1. 时空基线网络图
plot_baseline_network(MINTPY_DIR, EXPORT_DIR)

# 2. LOS 形变速率图 (左右布局: 速率 + 不确定性)
plot_velocity_map(MINTPY_DIR, EXPORT_DIR, coh_threshold=0.5)

# 3. 速率诊断散点图
plot_velocity_diagnostics(MINTPY_DIR, EXPORT_DIR, coh_threshold=0.5)

# 4. 逐景累积形变面板图 (精选 + 全时相)
plot_displacement_panels(MINTPY_DIR, EXPORT_DIR, coh_threshold=0.5)

# 5. 多代表点位移时序图 (定位图 + 时序曲线)
plot_point_timeseries(MINTPY_DIR, export_dir=EXPORT_DIR)

# 6. 导出 GeoTIFF / Shapefile / KMZ / CSV
export_geotiff(MINTPY_DIR, EXPORT_DIR)
export_shapefile(MINTPY_DIR, EXPORT_DIR, coh_threshold=0.7)
export_kmz(MINTPY_DIR, EXPORT_DIR, coh_threshold=0.7)
csv_df = export_csv(MINTPY_DIR, EXPORT_DIR, coh_threshold=0.7)
if csv_df is not None:
    display(csv_df.head(10))

# 7. 追加导出高可信成果与样本时序
depsi_like_export_summary = export_depsi_like_outputs(
    mintpy_dir=MINTPY_DIR,
    qc_report_dir=WORK_DIR / "mainchain_qc",
    export_dir=EXPORT_DIR,
)
display(depsi_like_export_summary)

# 8. 关键图件以缩略图库方式预览，避免 notebook 中逐张大图堆叠
fig_dir = EXPORT_DIR / "figures"
if not fig_dir.exists():
    fig_dir = EXPORT_DIR  # fallback if not yet organized
preview_figures = {
    "Baseline Network": fig_dir / "baseline_network.png",
    "Velocity And Uncertainty": fig_dir / "velocity_map.png",
    "Velocity Diagnostics": fig_dir / "velocity_diagnostics.png",
    "Cumulative Displacement (selected)": fig_dir / "displacement_panels.png",
    "Cumulative Displacement (all epochs)": fig_dir / "displacement_panels_all.png",
    "Representative Time Series": fig_dir / "timeseries_displacement.png",
    "DePSI-Like QC Panel": fig_dir / "depsi_quality_panel.png",
    "High-Confidence Time Series": fig_dir / "timeseries_samples_high_confidence.png",
}
display_notebook_figure_gallery(
    preview_figures,
    title="成果图预览",
    columns=2,
    max_height_px=360,
)

# 9. 整理 export 目录结构
organize_export_dir(EXPORT_DIR)

print(f"所有成果已导出至 {EXPORT_DIR}")
print("  figures/ — PNG (300 dpi) + PDF (矢量)")
print("  vector/  — SHP, KMZ, CSV")
print("  raster/  — GeoTIFF")
save_project_progress("export_done")


## Cell 8: 异常形变区自动识别

当前区域检测模块采用 `support_graph_v1` 的高可信点图区域分割方法。该方法在最终接受的 MintPy 时序和 DePSI-like 高可信骨架上识别连续异常形变区域，并输出 polygon、区域统计和区域级累计位移曲线。

### 1. 支持点集合与时间一致性特征

检测节点来自：

$$
\mathcal{V}=\{i\mid i\in strict \cup relaxed\}
$$

对每个支持点，代码直接从最终接受的 `rel0` 时序提取：

- `net_disp_mm`
- `peak_abs_disp_mm`
- `trend_mm_yr`
- `path_length_mm`
- `net_to_path_ratio`
- `sign_consistency`
- `pc1 / pc2 / pc3`
- `local_contrast_net / local_contrast_peak`
- `neighbor_trace_corr / support_density / qc_support`

其中关键一致性量是：

$$
path\_length = \sum_t |\Delta x_t|\cdot \mathbf{1}(|\Delta x_t|>0.25)
$$

$$
sign\_consistency = \frac{\max(N_+,N_-)}{N_+ + N_-}
$$

$$
net\_to\_path\_ratio = \frac{|x_T-x_0|}{path\_length}
$$

$$
temporal\_coherence = 0.5\cdot net\_to\_path\_ratio + 0.5\cdot sign\_consistency
$$

$$
qc\_support = 0.35\,ps\_score + 0.25\,tcoh + 0.20\,valid\_pair\_ratio + 0.20\,mainCC\_ratio
$$

### 2. 活动性、显著性和 support graph 边约束

记鲁棒 z-score 为 $\rho(\cdot)$，Sigmoid 为 $\sigma(\cdot)$。代码里的活动性与显著性写成：

$$
activity\_logit = 0.45\,\rho(peak\_abs\_disp) + 0.35\,\rho(|net\_disp|) + 0.20\,\rho(|trend|)
$$

$$
contrast\_logit = 0.60\,\rho(local\_contrast\_peak) + 0.40\,\rho(local\_contrast\_net)
$$

$$
activity = \sigma(activity\_logit)
$$

$$
saliency\_logit = 0.42\,activity\_logit + 0.24\,contrast\_logit + 1.10\,(temporal\_coherence-0.45)
+ 0.55\,(neighbor\_trace\_corr-0.35) + 0.35\,(support\_density-0.45) + 0.25\,(qc\_support-0.55)
$$

$$
saliency = \sigma(saliency\_logit)
$$

支持图默认 `graph_k = 12`，边只保留同时满足以下条件的相邻支持点：

$$
sign_i = sign_j,
\qquad
corr(x_i,x_j) \ge 0.58,
\qquad
d_{ij} \le \max(4.5\,d_{NN},\ 0.10\text{ km})
$$

### 3. Core/active 图生长与区域评分

活跃子图和核心子图是：

$$
active = \{i\mid saliency_i \ge \max(0.50,p_{80}),\ coherence_i\ge 0.48,\ qc\_support_i\ge 0.50\}
$$

$$
core = \{i\in active\mid saliency_i\ge \max(0.62,p_{92}),\ activity_i\ge p_{75},\ coherence_i\ge 0.55\}
$$

代码从 `core` 在 `active` 子图内做分量生长；没接触到核心的散点直接丢弃。每个候选区随后计算：

- `aoi_area_fraction`
- `internal_trace_corr`
- `temporal_coherence`
- `boundary_contrast`
- `compactness`
- `activity_level`

正式的区域评分是：

$$
region\_score = \sqrt{area\_fraction}\cdot internal\_trace\_corr\cdot temporal\_coherence\cdot boundary\_contrast\cdot \log(1+activity\_level)
$$

通过评分和面积门槛的候选区会继续做 outward growth，最后写出：

- `deformation_zones.geojson / shp / kmz`
- `deformation_zone_summary.json`
- `deformation_zone_timeseries.csv`
- `velocity_map_zones.png`

### 4. 区域级累计位移曲线

右侧区域曲线不再是单点，而是区域内支持点的 **20% trimmed mean**：

$$
x_{zone}(t)=\mathrm{TrimMean}_{20\%}\{x_i(t)\mid i\in zone\}
$$

并同时绘制 `p25-p75` 带状区间，因而更适合表达“区域整体形变演化”而不是单点抖动。


In [ ]:
import json
from pathlib import Path
from IPython.display import display
from insar_utils.deformation_zone import detect_deformation_zones
from insar_utils.viz import display_notebook_figure_gallery
from insar_utils.config import *

DEFORMATION_ZONE_CONFIG = {
    "enabled": True,
    "detector_mode": "support_graph_v1",
    "detection_domain": "hybrid",
    "zone_semantics": "anomalous_deformation",
    "output_geometry": "polygon",
    "forecast_point_scope": "all_high_confidence",
}

ZONE_MASK_PATH = None
deformation_zone_summary = None

if DEFORMATION_ZONE_CONFIG["enabled"]:
    deformation_zone_summary = detect_deformation_zones(
        MINTPY_DIR,
        qc_report_dir=WORK_DIR / "mainchain_qc",
        export_dir=EXPORT_DIR,
        detector_mode=DEFORMATION_ZONE_CONFIG["detector_mode"],
        detection_domain=DEFORMATION_ZONE_CONFIG["detection_domain"],
        zone_semantics=DEFORMATION_ZONE_CONFIG["zone_semantics"],
        output_geometry=DEFORMATION_ZONE_CONFIG["output_geometry"],
        forecast_point_scope=DEFORMATION_ZONE_CONFIG["forecast_point_scope"],
    )
    zone_summary_path = EXPORT_DIR / "deformation_zone_summary.json"
    if zone_summary_path.exists():
        deformation_zone_summary = json.loads(zone_summary_path.read_text(encoding="utf-8"))
    ZONE_MASK_PATH = EXPORT_DIR / "deformation_zone_mask.tif"
    display(deformation_zone_summary)
    if deformation_zone_summary is not None:
        print(
            f"检测到形变区数量: {deformation_zone_summary.get('n_detected_zones', 0)} | "
            f"status={deformation_zone_summary.get('status', 'unknown')}"
        )
    zone_figures = {
        "Velocity With Zones": EXPORT_DIR / "figures" / "velocity_map_zones.png",
    }
    display_notebook_figure_gallery(
        zone_figures,
        title="异常形变区检测预览",
        columns=1,
        max_height_px=420,
    )
    save_project_progress("deformation_zone_done")
else:
    print("已跳过异常形变区自动识别")


## Cell 9: 高可信点未来预测训练

当前预测模块采用共享数据层、`generic / hazard` 双模式建模和 conformal 区间校准。默认模式是 `generic`，默认预测范围为全部 DePSI-like 高可信点，而不是仅限于 zone 内样本。

### 1. 数据层：`robust_harmonic_v1` + 邻域图

每个点先统一到相对最早日期的累计位移：

$$
x_{rel0}(t)=x(t)-x(t_0)
$$

再在每个点上拟合同一套鲁棒谐波模型：

$$
x_{rel0}(t)=\beta_0+\beta_1 t + a_1\sin\frac{2\pi t}{365.25}+a_2\cos\frac{2\pi t}{365.25}+r(t)
$$

从而得到：

- `trend_component`
- `seasonal_component`
- `residual_component`

当前代码固定的 16 维序列通道分成 4 组：

- `raw`: `rel0 / delta / day_gap / sin_doy / cos_doy`
- `decomposition`: `trend_component / seasonal_component / residual_component / delta2 / rolling_velocity_3step / rolling_residual_std`
- `neighborhood`: `neighbor_mean_rel0 / neighbor_mean_delta / neighbor_delta_std`
- `event`: `local_event_persistence / abnormal_date_flag`

静态特征固定为：

- `ps_score`
- `tcoh`
- `valid_pair_ratio`
- `mainCC_ratio`
- `jump_risk`
- `anomaly_exposure`
- `velocity_mm_yr`
- `height_m`

预测邻域图与区域识别图不同，当前默认：

- `graph_k = 8`
- 边特征：`distance_m / elevation_diff_m / velocity_diff_mm_yr / ps_score_diff`

### 2. 点集策略与 forecast scope

当前代码先按 strict / relaxed 自动选点：

1. `strict_only`
2. `strict_plus_relaxed`
3. `fallback_baseline_only`

但默认预测范围是：

$$
forecast\_point\_scope = \texttt{all\_high\_confidence}
$$

因此区域识别结果默认只是附加信息，不会再次把预测缩成很少的 zone 内点。

### 3. `generic`：双候选模型并行训练后择优

`generic` 现在不是单一结构，而是并行训练两个真实模型：

1. `TemporalFusionForecaster`
2. `DecompTCNGRUQuantileForecaster`

#### 3.1 `DecompTCNGRUQuantileForecaster`

它把输入拆成长期分支、短期分支和静态分支：

$$
h_{long}=\mathrm{Attn}\big(\mathrm{TCN}(W_L x_{long})\big)
$$

$$
h_{short}=\mathrm{GRU}(W_S x_{short})
$$

$$
h_{static}=\mathrm{MLP}(s)
$$

代码中的门控融合写成：

$$
g = \sigma\big(W_g[h_{long},h_{short},h_{static}]\big)
$$

$$
h_{fused}=g\odot h_{long} + (1-g)\odot h_{short}
$$

最后把 `[h_{fused}, h_{static}]` 送入 quantile head 输出 `p10/p50/p90`。

#### 3.2 `TemporalFusionForecaster`

TFT 分支先做连续时间编码：

$$
z_t = W_x x_t + PE(day\_offset_t)
$$

再经过多层 self-attention，并用可学习时间门控做池化：

$$
\alpha_t = \mathrm{softmax}(w^\top z_t),
\qquad
h = \sum_t \alpha_t z_t
$$

如果启用邻域上下文，还会把邻居序列均值、邻居静态量和边特征拼接后做轻量 attention 聚合。

#### 3.3 generic canonical 选择规则

训练完成后，代码直接比较验证集 `MAE`，选择更优者作为 `generic_forecaster.pt`：

$$
model_{generic}=\arg\min_{m\in\{TFT,\ DecompTCNGRU\}} MAE_{val}(m)
$$

### 4. `hazard`：`GraphTCNAttnQuantileForecaster`

当 `forecast_mode="hazard"` 且图结构满足要求时，会额外训练图增强模型。它的每个时间步先做 edge-aware 邻域聚合：

$$
\alpha_{ij,t}=\mathrm{softmax}_j\,\phi\big([W_c x_{i,t},\ W_n x_{j,t},\ e_{ij},\ s_j]\big)
$$

$$
\tilde x_{i,t}=W_o\Big[x_{i,t},\ \sum_j \alpha_{ij,t}x_{j,t}\Big]
$$

随后：

$$
h_i = \mathrm{TCN}(\tilde x_i),
\qquad
u_i = \mathrm{TemporalAttentionPool}(h_i)
$$

再和静态特征融合输出 `p10/p50/p90`。最终是否真的启用 `hazard`，还要经过验证集稳定性判定：

- 如果 `hazard` 的 `coverage_error` 比 `generic` 多 `0.05` 以上，则退回 `generic`
- 否则如果 `hazard` 的验证 `RMSE` 至少改善 `2%`，才激活 `hazard`
- 否则保持 `generic` 更稳的选择

### 5. 训练目标：quantile pinball + monotonic penalty

代码里的正式损失函数是：

$$
\mathcal{L}_{quantile} = \frac{1}{3HT}\sum_{q\in\{0.1,0.5,0.9\}} \max\big(q(y-\hat y_q),(q-1)(y-\hat y_q)\big)
$$

再加上分位数单调约束：

$$
\mathcal{L}_{mono}=\mathrm{ReLU}(\hat y_{0.1}-\hat y_{0.5}) + \mathrm{ReLU}(\hat y_{0.5}-\hat y_{0.9})
$$

总损失是：

$$
\mathcal{L}=\mathcal{L}_{quantile}+0.1\,\mathcal{L}_{mono}
$$

当前 baseline 固定有 4 条：

- `persistence`
- `linear_trend`
- `seasonal_naive`
- `harmonic_trend`


In [ ]:
import json
from pathlib import Path
import torch
from IPython.display import display
from mintpy_forecast.train import train_forecast_models
from insar_utils.config import *

FORECAST_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FORECAST_TRAIN_CONFIG = {
    "enabled": True,
    "forecast_mode": "generic",
    "graph_k": 8,
    "decomposition": "robust_harmonic_v1",
    "calibrate_intervals": True,
    "zone_mask_path": ZONE_MASK_PATH,
    "forecast_point_scope": DEFORMATION_ZONE_CONFIG["forecast_point_scope"] if DEFORMATION_ZONE_CONFIG["enabled"] else "all_high_confidence",
    "lookback": 12,
    "forecast_horizon": 3,
    "min_points_for_training": 500,
    "min_train_windows": 3000,
    "min_val_windows": 500,
    "max_windows_train": 180000,
    "max_windows_val": 50000,
    "max_windows_test": 50000,
    "epochs": 20,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "patience": 5,
    "seed": 42,
    "device": FORECAST_DEVICE,
}

if FORECAST_TRAIN_CONFIG["enabled"]:
    print(f"预测训练 device: {FORECAST_TRAIN_CONFIG['device']}")
    forecast_train_summary = train_forecast_models(
        MINTPY_DIR,
        qc_report_dir=WORK_DIR / "mainchain_qc",
        output_dir=FORECAST_DIR,
        forecast_mode=FORECAST_TRAIN_CONFIG["forecast_mode"],
        graph_k=FORECAST_TRAIN_CONFIG["graph_k"],
        decomposition=FORECAST_TRAIN_CONFIG["decomposition"],
        calibrate_intervals=FORECAST_TRAIN_CONFIG["calibrate_intervals"],
        zone_mask_path=FORECAST_TRAIN_CONFIG["zone_mask_path"],
        forecast_point_scope=FORECAST_TRAIN_CONFIG["forecast_point_scope"],
        lookback=FORECAST_TRAIN_CONFIG["lookback"],
        forecast_horizon=FORECAST_TRAIN_CONFIG["forecast_horizon"],
        min_points_for_training=FORECAST_TRAIN_CONFIG["min_points_for_training"],
        min_train_windows=FORECAST_TRAIN_CONFIG["min_train_windows"],
        min_val_windows=FORECAST_TRAIN_CONFIG["min_val_windows"],
        max_windows_train=FORECAST_TRAIN_CONFIG["max_windows_train"],
        max_windows_val=FORECAST_TRAIN_CONFIG["max_windows_val"],
        max_windows_test=FORECAST_TRAIN_CONFIG["max_windows_test"],
        epochs=FORECAST_TRAIN_CONFIG["epochs"],
        batch_size=FORECAST_TRAIN_CONFIG["batch_size"],
        learning_rate=FORECAST_TRAIN_CONFIG["learning_rate"],
        patience=FORECAST_TRAIN_CONFIG["patience"],
        seed=FORECAST_TRAIN_CONFIG["seed"],
        device=FORECAST_TRAIN_CONFIG["device"],
    )
    display(forecast_train_summary)
    data_summary_path = Path(forecast_train_summary["data_summary_path"])
    if data_summary_path.exists():
        display(json.loads(data_summary_path.read_text(encoding="utf-8")))
    calibration_path = FORECAST_DIR / "forecast_calibration.json"
    if calibration_path.exists():
        print("===== 预测校准摘要 =====")
        display(json.loads(calibration_path.read_text(encoding="utf-8")))
    print(f"\n预测训练目录: {FORECAST_DIR}")
    save_project_progress("forecast_train_done")
else:
    print("已跳过预测训练")


## Cell 10: 高可信点未来预测推理、评估与绘图

这一单元直接消费 `Cell 9` 训练出的模型、normalizer、calibration 和 explainability 结果，统一完成：

- 最新窗口推理
- baseline / generic / hazard / active_model 统一评估
- 预测图件生成
- 预测点级清单导出

### 1. 双语义输出：offset 与 rel0

虽然模型内部训练的是 future offset，但对外会同时输出两种语义：

- `pred_offset_p10/p50/p90`
- `pred_rel0_p10/p50/p90`

两者关系是：

$$
\hat x_{rel0}(t+h)=x_{rel0}(t)+\hat y_{offset}(t\rightarrow t+h)
$$

新版 canonical 字段统一写成 **校准后区间**，原始分位数保存在 `*_raw` 里，仅用于诊断。

### 2. CQR conformal 校准

当前正式不确定性模式是：

$$
uncertainty\_mode = \texttt{cqr\_conformal\_v1}
$$

在验证集上，对每个 forecast horizon 单独计算 nonconformity：

$$
s_h^{(n)} = \max(q_{10,raw}^{(n)}-y^{(n)},\ y^{(n)}-q_{90,raw}^{(n)},\ 0)
$$

再取 split-conformal 修正量 $\delta_h$，并得到：

$$
q_{10,cal}=q_{10,raw}-\delta_h,
\qquad q_{50,cal}=q_{50,raw},
\qquad q_{90,cal}=q_{90,raw}+\delta_h
$$

当前目标覆盖率固定为：

$$
target\_coverage = 0.80
$$

因此 HDF5 中会同时保留：

- `predictions/pred_offset_p10|p50|p90`：校准后
- `predictions/pred_rel0_p10|p50|p90`：校准后
- `predictions/*_raw`：原始分位数
- `predictions/interval_width_raw`
- `predictions/interval_width_calibrated`

### 3. `c_pred` 是 risk-aware confidence surrogate

`c_pred` 不是模型直接输出，而是后处理得到的 calibrated confidence 代理量。代码里的定义是：

$$
width\_norm = \mathrm{normalize}(interval\_width_{cal})
$$

$$
c\_{pred} = \mathrm{clip}\Big(
0.60\,(1-width\_norm)
+0.20\,ps\_score
+0.10\,(1-jump\_risk)
+0.10\,(1-anomaly\_exposure)
\Big)\times reliability\_factor
$$

因此它表达的是“**校准后区间越窄、QC 越高、跳变和异常暴露越低，则预测越可靠**”。

### 4. 评估指标与切片评估

这一单元会统一计算：

- `mae_mm`
- `rmse_mm`
- `pinball_loss`
- `coverage`
- `coverage_error`
- `mean_interval_width_mm`
- `wis`

同时额外输出两类窗口切片：

- `event_windows`：预测 horizon 覆盖异常日期的窗口
- `stable_windows`：其余窗口

因此最终可以同时回答三类问题：

1. 主模型是否优于 `persistence / linear_trend / seasonal_naive / harmonic_trend`
2. 区间覆盖率是否接近 `0.80`
3. 异常期与稳定期的预测表现是否存在系统差异

### 5. 图件与库存

当前正式图件包括：

- `forecast_calibration_curve.png`
- `forecast_feature_group_importance.png`
- `forecast_confidence_map.png`
- `forecast_neighbor_influence_map.png`（仅 hazard）

同时会导出：

- `forecast_predictions.h5`
- `forecast_summary.csv`
- `forecast_point_inventory.csv`

因此这一格结束后，既有模型级评估，也有点级可追踪清单和地图级展示结果。


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
from mintpy_forecast.infer import run_forecast_inference
from mintpy_forecast.evaluate import evaluate_forecast_predictions
from mintpy_forecast.viz import generate_forecast_figures
from insar_utils.viz import display_notebook_figure_gallery
from insar_utils.config import *

FORECAST_INFER_CONFIG = {
    "enabled": True,
    "model_dir": FORECAST_DIR,
    "zone_mask_path": ZONE_MASK_PATH,
    "forecast_point_scope": DEFORMATION_ZONE_CONFIG["forecast_point_scope"] if DEFORMATION_ZONE_CONFIG["enabled"] else "all_high_confidence",
    "forecast_output": FORECAST_DIR / "forecast_predictions.h5",
    "summary_csv": FORECAST_DIR / "forecast_summary.csv",
    "evaluation": FORECAST_DIR / "forecast_evaluation.json",
    "baseline_csv": FORECAST_DIR / "forecast_baseline_comparison.csv",
    "point_inventory_csv": FORECAST_DIR / "forecast_point_inventory.csv",
    "figure_output_dir": FORECAST_DIR / "figures",
    "batch_size": 512,
    "device": "cuda" if __import__("torch").cuda.is_available() else "cpu",
}

if FORECAST_INFER_CONFIG["enabled"]:
    print(f"预测推理 device: {FORECAST_INFER_CONFIG['device']}")
    forecast_infer_summary = run_forecast_inference(
        MINTPY_DIR,
        qc_report_dir=WORK_DIR / "mainchain_qc",
        model_dir=FORECAST_INFER_CONFIG["model_dir"],
        output_path=FORECAST_INFER_CONFIG["forecast_output"],
        summary_csv_path=FORECAST_INFER_CONFIG["summary_csv"],
        point_inventory_path=FORECAST_INFER_CONFIG["point_inventory_csv"],
        batch_size=FORECAST_INFER_CONFIG["batch_size"],
        zone_mask_path=FORECAST_INFER_CONFIG["zone_mask_path"],
        forecast_point_scope=FORECAST_INFER_CONFIG["forecast_point_scope"],
        device=FORECAST_INFER_CONFIG["device"],
    )
    forecast_eval_summary = evaluate_forecast_predictions(
        MINTPY_DIR,
        qc_report_dir=WORK_DIR / "mainchain_qc",
        model_dir=FORECAST_INFER_CONFIG["model_dir"],
        output_path=FORECAST_INFER_CONFIG["evaluation"],
        baseline_csv_path=FORECAST_INFER_CONFIG["baseline_csv"],
        batch_size=FORECAST_INFER_CONFIG["batch_size"],
        zone_mask_path=FORECAST_INFER_CONFIG["zone_mask_path"],
        forecast_point_scope=FORECAST_INFER_CONFIG["forecast_point_scope"],
        device=FORECAST_INFER_CONFIG["device"],
    )
    forecast_figures = generate_forecast_figures(
        MINTPY_DIR,
        qc_report_dir=WORK_DIR / "mainchain_qc",
        forecast_path=FORECAST_INFER_CONFIG["forecast_output"],
        export_dir=FORECAST_INFER_CONFIG["figure_output_dir"],
        zone_mask_path=FORECAST_INFER_CONFIG["zone_mask_path"],
        forecast_point_scope=FORECAST_INFER_CONFIG["forecast_point_scope"],
    )
    display(forecast_infer_summary)
    display(forecast_eval_summary)

    data_summary_path = FORECAST_DIR / "forecast_data_summary.json"
    if data_summary_path.exists():
        print("===== 预测数据摘要 =====")
        display(json.loads(data_summary_path.read_text(encoding="utf-8")))

    train_summary_path = FORECAST_DIR / "forecast_train_summary.json"
    if train_summary_path.exists():
        train_summary_payload = json.loads(train_summary_path.read_text(encoding="utf-8"))
        print("===== 请求模式 / 实际模式 / fallback =====")
        display({
            "forecast_mode_requested": train_summary_payload.get("forecast_mode_requested"),
            "forecast_mode_actual": train_summary_payload.get("forecast_mode_actual"),
            "active_model": train_summary_payload.get("active_model"),
            "fallback_triggered": train_summary_payload.get("fallback_triggered"),
            "fallback_reasons": train_summary_payload.get("fallback_reasons"),
        })

    calibration_json = FORECAST_DIR / "forecast_calibration.json"
    if calibration_json.exists():
        print("===== 预测区间校准 =====")
        display(json.loads(calibration_json.read_text(encoding="utf-8")))

    baseline_csv = FORECAST_INFER_CONFIG["baseline_csv"]
    if Path(baseline_csv).exists():
        print("===== baseline / generic / hazard / active_model 对比（前 12 行） =====")
        display(pd.read_csv(baseline_csv).head(12))

    point_inventory_csv = FORECAST_INFER_CONFIG["point_inventory_csv"]
    if Path(point_inventory_csv).exists():
        print("===== 预测点级清单（前 10 行） =====")
        display(pd.read_csv(point_inventory_csv).head(10))

    preview_figures = {
        key.replace("_", " ").title(): Path(fig_path)
        for key, fig_path in sorted(forecast_figures.items())
    }
    display_notebook_figure_gallery(
        preview_figures,
        title="预测图件预览",
        columns=2,
        max_height_px=320,
    )

    print(f"预测结果: {FORECAST_INFER_CONFIG['forecast_output']}")
    print(f"预测评估报告: {FORECAST_INFER_CONFIG['evaluation']}")
    print(f"预测 baseline 对比: {FORECAST_INFER_CONFIG['baseline_csv']}")
    print(f"预测图件目录: {FORECAST_INFER_CONFIG['figure_output_dir']}")
    save_project_progress("forecast_infer_done")
else:
    print("已跳过预测推理与评估")


## Cell 11: 磁盘清理（可选）

这一单元只用于最终手动清理。当前主线已经不再包含深度学习时序矫正链，因此这里的清理只围绕主 InSAR 反演、QC、MintPy 与预测产物展开。前面步骤已经自动执行了分步清理，这一格主要用于最终释放剩余临时数据。


In [ ]:
from insar_utils.config import cleanup_temp

# 取消下方注释以执行清理 (删除 .zip + ISCE2 临时文件)
# cleanup_temp()